# YOLO26s: improving distant-deer detection

I'm keeping our YOLO26s detector and testing whether training crops and a gentle fine-tune help it find small deer. The earlier crop pilot improved recall on a BAMBI/decoy slice, but cost extra inference time and false boxes. This run tests a training change; it doesn't assume that earlier gain will carry over.

The server run was launched separately as a detached service. Leave START_TRAINING=False when using this notebook to inspect it; do not launch a duplicate run.

## What stays and what changes

Our original Deer v3 split stays at **3,193 train / 358 validation / 238 test**. It combines Young Deer, BAMBI, corrected Mississippi decoys, AWIR, Sherbrooke and reviewed aerial Rucervus. Training decoys keep 4x exposure. [Dataset preparation](../../docs/datasets/deer-v3.md) explains why we reduced repeated frames, corrected boxes and kept related scenes together.

This time I add training-only crops around small deer, capped at 100 per source. They give distant deer more pixels without requiring extra inference passes. All visible deer in a crop stay labelled. The runner skips crops that leave ambiguous slivers at the edge.

We fine-tune the saved deer checkpoint for **up to 25 epochs, 1280 input, LR 0.0001 and patience 7**. Batch 1 and FP32 are conservative choices for Michael's laptop after the earlier GPU failures. They aren't guaranteed to fix hardware/driver faults. The native best checkpoint is selected by validation fitness.

The test set has already been inspected in previous work. We'll use it only as a regression check after validation, and still need an independent camera/flight session.

In [ ]:
from pathlib import Path
import json, subprocess, sys, os
from IPython.display import display, Image

# Edit these for Michael's server. Keep this separate from Yash's run directories.
REPO = Path('/home/mrd/yolo26s_refinement/code').resolve()
DATA = Path('/home/mrd/deer-data/Deer Training Dataset v3').resolve()
WEIGHTS = Path('/home/mrd/yolo26s-deer-v3-best.pt').resolve()
RUN = Path('/home/mrd/yolo26s_refinement/run1').resolve()
SCRIPT = REPO / 'prototyping/yolo26s-refinement/run.py'

# Leave false until we've checked the paths, crop preview and available GPU.
START_TRAINING = False
RUN_TEST_REGRESSION = False

def run(mode, *extra):
    cmd = [sys.executable, '-u', str(SCRIPT), mode, '--data', str(DATA), '--run', str(RUN), *map(str, extra)]
    env = dict(os.environ, MPLBACKEND='Agg')
    with subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, env=env) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError(f'{mode} failed. Preserve outputs and inspect the error above.')

def show_json(path):
    display(json.loads(path.read_text()))

print('Notebook ready; no training started.')

In [ ]:
# Read-only progress for the detached server run.
log = RUN.parent / 'recovery-pipeline.log'
if not log.exists():
    log = RUN.parent / 'pipeline.log'
if log.exists():
    import re
    lines = [re.sub(r'\x1b\[[0-9;]*[A-Za-z]', '', x)
             for x in re.split(r'[\r\n]', log.read_text(errors='replace')) if x.strip()]
    print('\n'.join(lines[-12:]))
    print('Pipeline complete:', (RUN / 'pipeline_complete.json').exists())
else:
    print('Open on greglinux with the paths above to see the running job.')

## Prepare the training data

Use the extracted data and saved weights already on disk. This doesn't download anything or change the original images/labels. It writes derived crops and a training list in the new run folder. A rerun won't overwrite an existing preparation.

In [ ]:
if not (RUN / 'manifest.json').exists():
    run('prepare', '--weights', WEIGHTS)
show_json(RUN / 'manifest.json')
display(Image(filename=str(RUN / 'prepared/tile_review.jpg')))


## GPU smoke test and training

Check the crop labels above first. The smoke test uses its own short run and verifies that weights save. Full training starts from the original deer checkpoint, not the smoke weights. Don't run this alongside another GPU job.

For unattended training, use the equivalent CLI in tmux; closing an active notebook kernel can stop the job. Failures aren't silently restarted. If interrupted, use the documented `train --resume` command to restore the last full-training checkpoint and optimizer.

In [ ]:
if START_TRAINING:
    if not (RUN / 'smoke_complete.json').exists():
        run('smoke')
    if not (RUN / 'training_complete.json').exists():
        run('train')
else:
    print('Training disabled. Set START_TRAINING=True only when ready.')

## Loss, precision, recall and AP over epochs

Native plots show the optimization history. A lower training loss by itself doesn't mean better generalization. Look for improving validation metrics; if training keeps improving while validation gets worse, don't claim a better detector.

In [ ]:
plot = RUN / 'fit/results.png'
if plot.exists():
    display(Image(filename=str(plot)))
    import pandas as pd
    display(pd.read_csv(RUN / 'fit/results.csv').tail(10))
else:
    print('No training results yet.')

## Validation: can a human review the candidates?

The shared evaluator reports AP50/AP50–95, precision/recall/F1, count errors and results per source. Sherbrooke dominates the validation box count, so I also report mean recall across sources.

I want high recall, but a threshold close to zero produced far too many false boxes before. This report looks for 95% mean source recall under a budget of two false boxes/image. That's a workload assumption we can change, not a guarantee. It explicitly reports when the target isn't met. The older F1-calibrated threshold is kept separately for comparison.

Profiling measures batch-one FP32 memory and file-to-prediction latency on this GPU. It doesn't certify Jetson throughput, video latency or the combined detector/OCR/anomaly workload.

In [ ]:
if (RUN / 'training_complete.json').exists():
    run('evaluate')
    show_json(RUN / 'val/metrics.json')
    show_json(RUN / 'val/review_metrics.json')
    show_json(RUN / 'val/profile.json')
    display(Image(filename=str(RUN / 'val/threshold_curves.png')))
    display(Image(filename=str(RUN / 'val/review_tradeoff.png')))
else:
    print('Finish training before validation.')

## Historical test regression check

Run this once we've chosen the recipe and thresholds using validation. Test uses the frozen validation thresholds. Compare source slices, not just one pooled number. Repeated tuning against this test would make it less informative; the next honest field test needs a new held-out session.

In [ ]:
if RUN_TEST_REGRESSION:
    run('evaluate', '--test')
    show_json(RUN / 'test/metrics.json')
    show_json(RUN / 'test/review_metrics.json')
else:
    print('Test regression disabled until validation review is done.')

## What the new flight setup changes

The current team plan is a VTOL with a gimbal-mounted zoom camera, surveying and zooming for tags during the same pass. That means we need more distant and oblique examples, not just top-down images. Optical zoom can make deer/tags larger in the frame, but reduces area coverage. Training crops simulate scale changes; they don't simulate the full optics, motion blur or viewing angle.

Keep YOLO detecting during the survey and run tag/OCR/anomaly work on selected detailed frames. We still need human confirmation and stable deer IDs across frames. The 2D map is useful for the pilot and tracker team, but a detection isn't a GPS location: projection needs synchronized aircraft/gimbal pose, zoom calibration and a ground model. VSLAM is optional if the aircraft provides a usable georeferenced pose.

This notebook doesn't implement tag detection, tracking, geolocation or gimbal control.

## My conclusion after this run

*Fill this in after validation.* Did small-deer recall improve at a tolerable review workload? Which sources still fail? Is native inference fast enough, or do selective crops help enough to justify extra latency?

Weights, optimizer checkpoints, configuration, training plots, prediction JSON, source reports and profiles stay in the run directory. Keep that directory and the original data outside Git. We can add this interpretation without rerunning training.